# 질병청 CHS SAS 데이터 사용 가능성 점검

`Data` 폴더에 있는 2020~2025년 `.sas7bdat` 파일을 열어보고, 지금까지 만든 설문 기반 건강 예측 모델에 사용할 수 있는지 확인합니다.

확인 결과 요약:

- 2020~2025년 파일 6개 모두 pandas `read_sas`로 정상 로딩됩니다.
- 전체 규모는 약 138.5만 명입니다.
- 6개 연도에 공통으로 존재하는 컬럼은 102개입니다.
- 당뇨 예측에는 사용할 가능성이 높습니다. `dia_*` 진단/관리 변수와 흡연, 음주, 신체활동, 영양, 정신건강, BMI/체중 관련 설문 변수가 있습니다.
- 비만 예측에도 사용할 수 있습니다. 다만 BMI 변수 구조가 연도별로 다릅니다. 2020/2023/2024는 `oba_bmi`가 있고, 2021~2025는 키/몸무게로 보이는 `oba_02z1`, `oba_03z1`이 있습니다.
- 수면장애 예측에는 바로 쓰기 어렵습니다. `Sleep Disorder`, `Insomnia`, `Sleep Apnea`처럼 명시적인 수면장애 타깃 컬럼은 보이지 않았습니다.
- 변수명이 코드형이라 공식 조사표/코드북이 꼭 필요합니다. 특히 `7`, `8`, `9`, `77`, `88`, `99`, `77777`, `88888`, `99999` 같은 무응답/비해당 코드를 결측 처리해야 합니다.


In [1]:
from pathlib import Path
from collections import Counter

import pandas as pd

DATA_DIR = Path("../Data")
sas_files = sorted(DATA_DIR.glob("*.sas7bdat"))

sas_files


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


[PosixPath('../Data/chs20_all_re.sas7bdat'),
 PosixPath('../Data/chs21_all.sas7bdat'),
 PosixPath('../Data/chs22_all.sas7bdat'),
 PosixPath('../Data/chs23_all.sas7bdat'),
 PosixPath('../Data/chs24_all.sas7bdat'),
 PosixPath('../Data/chs25_all.sas7bdat')]

## 1. SAS 파일 메타데이터 확인

파일이 크기 때문에 `chunksize`로 나누어 읽으면서 행 수와 컬럼 수만 먼저 확인합니다.


In [2]:
def read_sas_chunks(path, chunksize=50_000):
    return pd.read_sas(
        path,
        format="sas7bdat",
        encoding="euc-kr",
        iterator=True,
        chunksize=chunksize,
    )


def inspect_sas_file(path):
    rows = 0
    columns = None

    for chunk in read_sas_chunks(path):
        chunk.columns = [col.lower() for col in chunk.columns]
        if columns is None:
            columns = list(chunk.columns)
        rows += len(chunk)

    return {
        "file": path.name,
        "rows": rows,
        "columns": len(columns),
        "column_list": columns,
    }


metadata = [inspect_sas_file(path) for path in sas_files]
summary_df = pd.DataFrame(
    [{k: v for k, v in item.items() if k != "column_list"} for item in metadata]
)

summary_df


,file,rows,columns
0,chs20_all_re.sas7bdat,229269,171
1,chs21_all.sas7bdat,229242,193
2,chs22_all.sas7bdat,231785,161
3,chs23_all.sas7bdat,231752,178
4,chs24_all.sas7bdat,231728,209
5,chs25_all.sas7bdat,231615,201


In [3]:
print("총 행 수:", summary_df["rows"].sum())
print("연도별 컬럼 수 범위:", summary_df["columns"].min(), "~", summary_df["columns"].max())


총 행 수: 1385391
연도별 컬럼 수 범위: 161 ~ 209


## 2. 연도별 공통 컬럼 확인

연도별 조사 문항이 조금씩 달라지므로, 6개 연도를 통합하려면 공통 컬럼을 우선 사용하는 것이 안전합니다.


In [4]:
cols_by_file = {item["file"]: set(item["column_list"]) for item in metadata}

common_cols = sorted(set.intersection(*cols_by_file.values()))
union_cols = sorted(set.union(*cols_by_file.values()))

print("공통 컬럼 수:", len(common_cols))
print("전체 컬럼 수:", len(union_cols))

common_cols[:120]


공통 컬럼 수: 102
전체 컬럼 수: 356


['age',
 'ctprvn_code',
 'cva_11z2',
 'cva_12z1',
 'cva_14z2',
 'cva_16z2',
 'cva_17z1',
 'dia_04z1',
 'dia_13a2',
 'dia_13b2',
 'dia_13c2',
 'dia_14z1',
 'dia_15z1',
 'dia_19z1',
 'dia_22z2',
 'dong_ty_code',
 'dra_01z1',
 'drb_01z3',
 'drb_03z1',
 'drb_04z1',
 'drb_05z1',
 'exmprs_no',
 'fma_04z1',
 'fma_12z1',
 'fma_13z1',
 'fma_14z1',
 'fma_19z3',
 'fma_24z2',
 'house_ty_code',
 'hshld_code',
 'hya_04z1',
 'hya_14a2',
 'hya_14b2',
 'hya_14c2',
 'hya_19z1',
 'i1a_04z2',
 'i2a_04z2',
 'i3a_04z2',
 'i4a_04z2',
 'i5a_04z2',
 'ira_01z1',
 'ira_02z1',
 'kstrata',
 'mbhld_co',
 'mbhld_code',
 'mta_01z1',
 'mtb_01z1',
 'mtb_02z1',
 'mtb_07a1',
 'mtb_07b1',
 'mtb_07c1',
 'mtb_07d1',
 'mtb_07e1',
 'mtb_07f1',
 'mtb_07g1',
 'mtb_07h1',
 'mtb_07i1',
 'mya_10z2',
 'mya_11z2',
 'mya_12z2',
 'mya_14z1',
 'mya_15z1',
 'nua_01z2',
 'nuc_01z2',
 'nuc_02z1',
 'nuc_03z1',
 'obb_01z1',
 'ora_01z1',
 'orb_01z1',
 'pbhlth_code',
 'pha_04z1',
 'pha_05z1',
 'pha_06z1',
 'pha_07z1',
 'pha_08z1',
 'pha_09z1'

## 3. 주요 변수군 존재 여부

질병청 지역사회건강조사(CHS) 변수명은 코드형입니다. 아래는 변수명 prefix 기준으로 당뇨, 비만, 흡연, 음주, 신체활동, 정신건강 등 후보 변수군이 각 연도에 존재하는지 보는 표입니다.


In [5]:
candidate_prefixes = {
    "diabetes_diagnosis_or_management": "dia",
    "obesity_bmi_or_body_size": "oba",
    "weight_control": "obb",
    "smoking": "sma",
    "drinking_a": "dra",
    "drinking_b": "drb",
    "physical_activity_a": "pha",
    "physical_activity_b": "phb",
    "nutrition_a": "nua",
    "nutrition_c": "nuc",
    "mental_health": "mt",
    "hypertension": "hya",
    "quality_or_health_status": "qoa",
    "subjective_health": "sra",
}

presence_rows = []

for item in metadata:
    file = item["file"]
    cols = item["column_list"]
    row = {"file": file}
    for group, prefix in candidate_prefixes.items():
        hits = [col for col in cols if col.startswith(prefix)]
        row[group] = len(hits)
    presence_rows.append(row)

presence_df = pd.DataFrame(presence_rows)
presence_df


,file,diabetes_diagnosis_or_management,obesity_bmi_or_body_size,weight_control,smoking,drinking_a,drinking_b,physical_activity_a,physical_activity_b,nutrition_a,nutrition_c,mental_health,hypertension,quality_or_health_status,subjective_health
0,chs20_all_re.sas7bdat,9,1,1,4,1,4,6,3,1,3,15,6,1,2
1,chs21_all.sas7bdat,10,3,1,4,2,5,8,3,1,3,21,7,1,2
2,chs22_all.sas7bdat,10,3,1,6,1,5,6,3,1,3,21,7,1,2
3,chs23_all.sas7bdat,12,4,1,7,3,7,7,3,1,3,14,9,1,2
4,chs24_all.sas7bdat,13,4,11,7,1,5,7,3,1,3,43,11,1,2
5,chs25_all.sas7bdat,12,3,1,7,1,5,7,3,1,3,23,9,4,2


In [6]:
def columns_starting_with(prefix):
    rows = []
    for item in metadata:
        cols = [col for col in item["column_list"] if col.startswith(prefix)]
        rows.append({"file": item["file"], "columns": cols, "n": len(cols)})
    return pd.DataFrame(rows)


columns_starting_with("dia")


,file,columns,n
0,chs20_all_re.sas7bdat,"[dia_19z1, dia_04z1, dia_05z1, dia_13c2, dia_1...",9
1,chs21_all.sas7bdat,"[dia_19z1, dia_04z1, dia_05z1, dia_13c2, dia_1...",10
2,chs22_all.sas7bdat,"[dia_19z1, dia_04z1, dia_05z1, dia_13c2, dia_1...",10
3,chs23_all.sas7bdat,"[dia_19z1, dia_04z1, dia_13c2, dia_13b2, dia_1...",12
4,chs24_all.sas7bdat,"[dia_19z1, dia_04z1, dia_18z1, dia_13c2, dia_1...",13
5,chs25_all.sas7bdat,"[dia_19z1, dia_04z1, dia_13c2, dia_13b2, dia_1...",12


In [7]:
columns_starting_with("oba")


,file,columns,n
0,chs20_all_re.sas7bdat,[oba_bmi],1
1,chs21_all.sas7bdat,"[oba_02z1, oba_03z1, oba_01z1]",3
2,chs22_all.sas7bdat,"[oba_02z1, oba_03z1, oba_01z1]",3
3,chs23_all.sas7bdat,"[oba_02z1, oba_03z1, oba_bmi, oba_01z1]",4
4,chs24_all.sas7bdat,"[oba_02z1, oba_03z1, oba_bmi, oba_01z1]",4
5,chs25_all.sas7bdat,"[oba_02z1, oba_03z1, oba_01z1]",3


## 4. 후보 핵심 변수 분포 확인

아래 변수들은 현재 목적에 특히 중요해 보이는 후보입니다.

- `dia_19z1`, `dia_04z1`: 당뇨 관련 후보 타깃/문항
- `oba_bmi`: BMI 값 후보
- `oba_01z1`, `oba_02z1`, `oba_03z1`: 비만도/키/몸무게 관련 후보
- `obb_01z1`: 체중 조절 관련 후보
- `sma_01z1`: 흡연 관련 후보
- `dra_01z1`: 음주 관련 후보
- `pha_04z1`: 신체활동 관련 후보
- `qoa_01z1`, `sra_01z3`: 삶의 질/주관적 건강 관련 후보

정확한 의미와 코드값은 공식 코드북으로 확정해야 합니다.


In [8]:
key_columns = [
    "age",
    "sex",
    "dia_19z1",
    "dia_04z1",
    "oba_bmi",
    "oba_01z1",
    "oba_02z1",
    "oba_03z1",
    "obb_01z1",
    "sma_01z1",
    "dra_01z1",
    "pha_04z1",
    "qoa_01z1",
    "sra_01z3",
]


def count_key_values(path, columns):
    counters = {col: Counter() for col in columns}
    present = {col: False for col in columns}

    for chunk in read_sas_chunks(path):
        chunk.columns = [col.lower() for col in chunk.columns]
        for col in columns:
            if col not in chunk.columns:
                continue
            present[col] = True
            if col in {"age", "oba_bmi"}:
                continue
            counters[col].update(chunk[col].dropna().astype(str).value_counts().to_dict())

    rows = []
    for col in columns:
        if not present[col]:
            continue
        rows.append(
            {
                "column": col,
                "top_values": counters[col].most_common(10) if counters[col] else "numeric/continuous",
            }
        )
    return pd.DataFrame(rows)


# 예시: 2025년 파일 기준 후보 변수 분포
count_key_values(DATA_DIR / "chs25_all.sas7bdat", key_columns)


,column,top_values
0,age,numeric/continuous
1,sex,"[(2.0, 124835), (1.0, 106780)]"
2,dia_19z1,"[(2.0, 146604), (1.0, 84118), (9.0, 892), (7.0..."
3,dia_04z1,"[(2.0, 197296), (1.0, 34295), (9.0, 23), (7.0,..."
4,oba_01z1,"[(3.0, 103086), (4.0, 77044), (2.0, 30600), (5..."
5,oba_02z1,"[(160.0, 17467), (170.0, 14235), (158.0, 14140..."
6,oba_03z1,"[(60.0, 14958), (70.0, 12323), (65.0, 11288), ..."
7,obb_01z1,"[(1.0, 90849), (4.0, 82063), (2.0, 45791), (3...."
8,sma_01z1,"[(8.0, 139689), (2.0, 85744), (1.0, 5745), (3...."
9,dra_01z1,"[(1.0, 195438), (2.0, 36174), (9.0, 2), (7.0, 1)]"


## 5. 사용 가능성 판단

### 당뇨 예측

사용 가능성이 높습니다.

- `dia_*` 컬럼이 모든 연도에 존재합니다.
- `dia_19z1`, `dia_04z1`, `dia_13a2` 등은 2020~2025년 공통 컬럼에 포함됩니다.
- 흡연(`sma_*`), 음주(`dra_*`, `drb_*`), 신체활동(`pha_*`, `phb_*`), 영양(`nua_*`, `nuc_*`), 정신건강(`mt*`), 주관적 건강(`sra_*`), 나이/성별 등 설문 기반 predictor도 풍부합니다.
- 단, 고혈압(`hya_*`), 심뇌혈관/질환 진단(`cva_*`, `mya_*` 등), 당뇨 관리/치료 문항은 predictor로 쓰면 병원 진단 정보가 섞일 수 있으므로 제외해야 합니다.

추천 방향: `dia_19z1` 또는 공식 코드북상 당뇨 의사진단 여부에 해당하는 변수를 타깃으로 두고, 진단/치료/검사 문항을 제외한 생활습관 설문 변수만 predictor로 사용합니다.

### 비만 예측

사용 가능합니다. 다만 연도별 BMI 관련 컬럼 구조를 맞춰야 합니다.

- 2020, 2023, 2024에는 `oba_bmi`가 존재합니다.
- 2021~2025에는 `oba_02z1`, `oba_03z1`이 존재하며 값 분포상 키(cm), 몸무게(kg) 후보로 보입니다.
- `oba_01z1`은 2021~2025에 존재하며 비만도 범주 후보로 보입니다.
- 2020은 `oba_bmi`만 있고 `oba_01z1`, `oba_02z1`, `oba_03z1`이 없어 통합 기준을 따로 정해야 합니다.

추천 방향: 2021~2025는 키/몸무게에서 BMI를 계산하거나 `oba_01z1`을 타깃으로 쓰고, 2020은 `oba_bmi`로 BMI 범주를 직접 생성해 맞춥니다.

### 수면장애 예측

현재 파일만으로는 제한적입니다.

- `sleep`, `insomnia`, `apnea`처럼 명시적인 수면장애 타깃에 해당하는 컬럼명은 확인되지 않았습니다.
- `qoa_*`, `sra_*`, `mt*` 같은 삶의 질/정신건강/주관적 건강 문항은 있지만, 이전 `Sleep_health_and_lifestyle_dataset.csv`처럼 수면장애 유형을 직접 예측하기는 어렵습니다.

추천 방향: 질병청 원자료 코드북에서 수면시간, 수면만족, 불면 관련 문항이 실제로 있는지 확인한 뒤 진행합니다. 없다면 수면장애 모델에는 별도 수면 데이터셋을 유지하는 편이 낫습니다.

## 최종 결론

이 CHS 데이터는 당뇨와 비만 예측에는 충분히 사용할 수 있습니다. 특히 표본 수가 크고 국내 데이터라 기존 외국 공개 데이터보다 프로젝트 목적에 더 잘 맞을 수 있습니다. 다만 변수명이 코드형이므로, 다음 단계는 공식 코드북으로 타깃 변수와 설문 predictor의 의미를 확정하고 무응답/비해당 코드를 결측 처리하는 것입니다.
